# 03a · The PRAGMA-like encoder, against Blueprint v2

[`pragma_like`](../../recipes/pragma_like) reimplements the
encoder of [PRAGMA](https://arxiv.org/abs/2604.08649) (Revolut). It shares
slots 0 and 1 with Blueprint v2: the same pragmatiq book, cutoffs, label
windows, and key-value-time tokens. Read
[notebooks 01–02](01_data_and_labels.ipynb) for those.
This notebook covers the three slots where it differs:

| Slot | Blueprint v2 | PRAGMA-style |
|---|---|---|
| 2 Sequence architecture | causal decoder over event summaries | **bidirectional** history encoder over event summaries |
| 3 Objective | next event's fields (+ MLM) | masked tokens, events, and keys (MLM) |
| 4 Readout | state after the last event | the `[USR]` slot of the history encoder |

Runs on CPU in under a minute on a generated synthetic book.

In [ ]:
# SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# SPDX-License-Identifier: Apache-2.0

from tfm.config import load_recipe_config
from tfm.runtime.common import recipe_dir, repo_root
from tfm.runtime.pipeline import setup_run
from tfm.training.reproducibility import seed_everything

config = load_recipe_config(recipe_dir("pragma_like") / "config.smoke.yaml")
seed_everything(config.seed)
run = setup_run(
    config=config,
    run_dir=repo_root() / "outputs" / "notebooks" / "pragma_like",
    recipe="pragma_like",
)
c = config.model.composition
print(c.granularity.type, "|", c.position, "|", c.objective, "|", c.readout)
for name, module in run.model.named_children():
    print(f"{name:<20} {sum(p.numel() for p in module.parameters()):>9,} parameters")

## Why the slots move together

A bidirectional history encoder lets every slot see every event, so it cannot
be trained to predict the next event: the answer would be in view (rule 1).
It is trained by masking instead (rule 2). And because every slot sees the
whole history, `[USR]` at slot 0 is a valid embedding here, while under
Blueprint v2's causal decoder slot 0 has seen only the profile (rule 3).
The kit checks all three:

In [ ]:
from tfm import EagerAttention, LastTokenPool
from tfm.contracts.errors import IncompatibleCompositionError
from tfm.experimental import (
    NextEventFields, ProfileEventDecoder, ProfileHistory, TimeRoPE, TokenEventKeyMLM,
    UsrTokenPool, validate_composition,
)

validate_composition(ProfileHistory(vocab_size=64), TimeRoPE(), EagerAttention(),
                     TokenEventKeyMLM(), UsrTokenPool(), None)
print("ProfileHistory + MLM + [USR]: accepted")
for label, args in {
    "ProfileHistory + next-event": (ProfileHistory(vocab_size=64), NextEventFields(), LastTokenPool()),
    "ProfileEventDecoder + [USR]": (ProfileEventDecoder(vocab_size=64), NextEventFields(), UsrTokenPool()),
}.items():
    structure, objective, readout = args
    try:
        validate_composition(structure, TimeRoPE(), EagerAttention(), objective, readout, None)
    except IncompatibleCompositionError as error:
        print(f"{label}: refused, {str(error).split(':')[0]}")

## Pretrain and probe

The same runtime as every other recipe: `run_train`, then `run_evaluate` on
the entity-disjoint split. Fixture numbers only show the wiring.

In [ ]:
from IPython.display import Markdown

from tfm.evaluation.reporting.report import metrics_markdown
from tfm.runtime.pipeline import run_evaluate, run_train

stats = run_train(run)
print({k: stats[k] for k in ("first_loss", "final_loss", "steps")})
Markdown(metrics_markdown(run_evaluate(run)))

## Going further

- `config.pragma_s/m/l.yaml` follow the PRAGMA-S/M/L size ladder of
  [dynamiq-ai/pragmatiq](https://github.com/dynamiq-ai/pragmatiq); none has a
  recorded result yet.
- `fraud_task.finetune` adds LoRA or full fine-tuning beside the frozen probe.
- [building-block ablation](../../benchmarks/ablations/building_blocks)
  measures one slot against another on shared splits.

Not bit-exact with PRAGMA or with pragmatiq: text is byte-level, not BPE, and
the Muon optimizer is not implemented.